# Fase C - Consultas SQL analíticas
## Data Warehouse de hurtos por modalidades

Este notebook ejecuta seis consultas SQL contra la bodega SQLite construida en la Fase B. Cada consulta responde una pregunta de negocio.

In [12]:
from pathlib import Path
import sqlite3
import pandas as pd

rutas_db = [Path('hurto_dw.db'), Path('entrega/hurto_dw.db')]
ruta_db = next((ruta for ruta in rutas_db if ruta.exists()), None)
if ruta_db is None:
    raise FileNotFoundError('No se encontró hurto_dw.db. Ejecute primero la Fase B.')

conn = sqlite3.connect(ruta_db)
print(f'Conectado a: {ruta_db.resolve()}')
print(pd.read_sql("SELECT name AS tabla FROM sqlite_master WHERE type = 'table' ORDER BY name", conn))

Conectado a: C:\Users\david\Desktop\Laboratorio_CienciaDatos\entrega\hurto_dw.db
            tabla
0  dim_arma_medio
1       dim_fecha
2   dim_municipio
3  dim_tipo_hurto
4     dim_victima
5     fact_hurtos


## 1. Evolución mensual de los hurtos

**Pregunta de negocio:** ¿Cómo ha evolucionado mensualmente la cantidad total de hurtos registrados en el periodo histórico?

In [13]:
sql_1 = """
SELECT f.año_mes AS periodo,
       SUM(h.cantidad) AS total_hurtos,
       COUNT(*) AS registros
FROM fact_hurtos AS h
JOIN dim_fecha AS f ON h.fecha_key = f.fecha_key
GROUP BY f.año_mes
ORDER BY f.año_mes;
"""
resultado_1 = pd.read_sql(sql_1, conn)
resultado_1

,periodo,total_hurtos,registros
0,2010-01,1916,1916
1,2010-02,1771,1771
2,2010-03,1990,1990
3,2010-04,1859,1859
4,2010-05,1616,1616
...,...,...,...
194,2026-03,3500,3500
195,2026-04,3239,3239
196,2026-05,3392,3392
197,2026-06,3416,3416


**Interpretación:** La tabla muestra la tendencia mensual de los casos y permite identificar periodos de mayor o menor incidencia. La columna `total_hurtos` es la métrica principal para estudiar y construir posteriormente una serie temporal de predicción.

## 2. Departamentos con mayor concentración de hurtos

**Pregunta de negocio:** ¿Qué departamentos concentran la mayor cantidad de hurtos y qué participación representan del total nacional?

In [14]:
sql_2 = """
WITH total_nacional AS (
    SELECT SUM(cantidad) AS total
    FROM fact_hurtos
)
SELECT m.departamento,
       SUM(h.cantidad) AS total_hurtos,
       ROUND(100.0 * SUM(h.cantidad) / total_nacional.total, 2) AS participacion_pct
FROM fact_hurtos AS h
JOIN dim_municipio AS m ON h.municipio_key = m.municipio_key
CROSS JOIN total_nacional
GROUP BY m.departamento
ORDER BY total_hurtos DESC
LIMIT 10;
"""
resultado_2 = pd.read_sql(sql_2, conn)
resultado_2

,departamento,total_hurtos,participacion_pct
0,ANTIOQUIA,144040,21.94
1,BOGOTA,113399,17.27
2,VALLE,104537,15.92
3,CAUCA,36692,5.59
4,ATLÁNTICO,33000,5.03
5,HUILA,20806,3.17
6,NARIÑO,19534,2.97
7,NORTE DE SANTANDER,18719,2.85
8,CESAR,18497,2.82
9,SANTANDER,17530,2.67


**Interpretación:** Los diez departamentos ordenados en la tabla concentran la mayor cantidad de casos reportados. La participación porcentual facilita priorizar recursos de prevención y muestra si el fenómeno está distribuido ampliamente o concentrado en pocos territorios.

## 3. Modalidad de hurto predominante

**Pregunta de negocio:** ¿Predominan los hurtos de motocicletas o los hurtos de automotores en el conjunto histórico?

In [15]:
sql_3 = """
SELECT t.tipo_vehiculo AS modalidad,
       SUM(h.cantidad) AS total_hurtos,
       COUNT(DISTINCT h.municipio_key) AS municipios_afectados
FROM fact_hurtos AS h
JOIN dim_tipo_hurto AS t ON h.tipo_hurto_key = t.tipo_hurto_key
GROUP BY t.tipo_vehiculo
ORDER BY total_hurtos DESC;
"""
resultado_3 = pd.read_sql(sql_3, conn)
resultado_3

,modalidad,total_hurtos,municipios_afectados
0,MOTOCICLETAS,497736,2038
1,AUTOMOTORES,158899,1354


**Interpretación:** La modalidad que aparece en la primera fila es la de mayor volumen histórico y también puede compararse por su alcance territorial. Esta diferencia orienta la pregunta de predicción del proyecto hacia la modalidad con más observaciones disponibles.

## 4. Medios o armas más reportados

**Pregunta de negocio:** ¿Cuáles son los cinco medios o armas más reportados en los casos de hurto?

In [16]:
sql_4 = """
SELECT a.arma_medio,
       a.categoria_arma,
       SUM(h.cantidad) AS total_hurtos,
       COUNT(DISTINCT h.municipio_key) AS municipios_afectados
FROM fact_hurtos AS h
JOIN dim_arma_medio AS a ON h.arma_medio_key = a.arma_medio_key
GROUP BY a.arma_medio_key, a.arma_medio, a.categoria_arma
ORDER BY total_hurtos DESC
LIMIT 5;
"""
resultado_4 = pd.read_sql(sql_4, conn)
resultado_4

,arma_medio,categoria_arma,total_hurtos,municipios_afectados
0,LLAVE MAESTRA,Herramienta / ganzúa,225206,1279
1,SIN EMPLEO DE ARMAS,Sin empleo de armas,193700,1484
2,ARMA DE FUEGO,Arma de fuego,190560,1637
3,ARMA BLANCA / CORTOPUNZANTE,Arma blanca / cortopunzante,16279,755
4,NO REPORTADO,No reportado / otro,14820,732


**Interpretación:** El resultado identifica los medios con mayor frecuencia y su categoría agrupada para facilitar decisiones operativas. También permite observar si un medio es frecuente en muchos municipios o si está concentrado en pocos territorios.

## 5. Perfil demográfico de las víctimas

**Pregunta de negocio:** ¿Qué combinaciones de género y grupo etario concentran más víctimas de hurto?

In [17]:
sql_5 = """
SELECT v.genero,
       v.grupo_etario,
       SUM(h.cantidad) AS total_hurtos,
       COUNT(DISTINCT h.municipio_key) AS municipios_afectados
FROM fact_hurtos AS h
JOIN dim_victima AS v ON h.victima_key = v.victima_key
GROUP BY v.genero, v.grupo_etario
ORDER BY total_hurtos DESC;
"""
resultado_5 = pd.read_sql(sql_5, conn)
resultado_5

,genero,grupo_etario,total_hurtos,municipios_afectados
0,MASCULINO,ADULTOS,526608,2114
1,FEMENINO,ADULTOS,125023,1286
2,MASCULINO,ADOLESCENTES,2926,379
3,FEMENINO,ADOLESCENTES,704,170
4,NO REPORTADO,ADULTOS,560,149
5,MASCULINO,MENORES,494,99
6,NO REPORTADO,NO REPORTADO,205,88
7,FEMENINO,MENORES,108,49
8,NO REPORTADO,ADOLESCENTES,7,7


**Interpretación:** La combinación ubicada en la primera fila representa el perfil con mayor cantidad de registros en la bodega. Las categorías `NO REPORTADO` deben considerarse al interpretar el resultado, porque reflejan una limitación de completitud del registro y no necesariamente un perfil real de las víctimas.

## 6. Municipios con mayor volumen de hurtos de motocicletas

**Pregunta de negocio:** ¿Cuáles son los diez municipios con mayor cantidad histórica de hurtos de motocicletas?

In [18]:
sql_6 = """
SELECT ROW_NUMBER() OVER (ORDER BY SUM(h.cantidad) DESC) AS posicion,
       m.departamento,
       m.municipio,
       SUM(h.cantidad) AS hurtos_motocicletas,
       COUNT(DISTINCT f.año) AS años_con_registro
FROM fact_hurtos AS h
JOIN dim_municipio AS m ON h.municipio_key = m.municipio_key
JOIN dim_tipo_hurto AS t ON h.tipo_hurto_key = t.tipo_hurto_key
JOIN dim_fecha AS f ON h.fecha_key = f.fecha_key
WHERE t.tipo_vehiculo = 'MOTOCICLETAS'
GROUP BY m.municipio_key, m.departamento, m.municipio
ORDER BY hurtos_motocicletas DESC
LIMIT 10;
"""
resultado_6 = pd.read_sql(sql_6, conn)
resultado_6

,posicion,departamento,municipio,hurtos_motocicletas,años_con_registro
0,1,ANTIOQUIA,Medellín (CT),78821,17
1,2,BOGOTA,Bogotá D.C. (CT),60259,17
2,3,VALLE,Cali (CT),47504,17
3,4,ATLÁNTICO,Barranquilla (CT),13834,17
4,5,CAUCA,Popayán (CT),13567,17
5,6,NORTE DE SANTANDER,Cúcuta (CT),9790,17
6,7,ANTIOQUIA,Bello,9488,17
7,8,META,Villavicencio (CT),9035,17
8,9,NARIÑO,Pasto (CT),8277,17
9,10,CESAR,Valledupar (CT),7679,17


**Interpretación:** Estos municipios presentan la mayor carga histórica de hurtos de motocicletas, por tanto, son candidatos prioritarios para análisis predictivo y acciones focalizadas; `años_con_registro` agrega contexto temporal para distinguir entre una incidencia concentrada y otra sostenida.

## Cierre

Las seis consultas utilizan la tabla de hechos y las dimensiones de tiempo, territorio, modalidad, medio y víctima, en conjunto, permiten pasar de un conteo general a una lectura temporal, geográfica, operativa y sociodemográfica de los hurtos.

In [ ]:
conn.close()
print('Conexión SQLite cerrada correctamente.')